# SPY/VIX Z-Score Backtest

**Strategy:**
- **Enter Long:** When VIX Z-score rises above moving average from below -1
- **Exit Long:** When VIX Z-score crosses below moving average above 1

Based on the ThetaTrend VIX Z-Score indicator

## Install Dependencies

In [ ]:
!pip install yfinance pandas numpy matplotlib

## Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')

## Backtest Configuration

In [ ]:
# Backtest parameters
START_DATE = '2010-01-01'
END_DATE = datetime.today().strftime('%Y-%m-%d')
INITIAL_CAPITAL = 100000

# VIX Z-Score parameters (from ThinkScript)
VIX_SHORT = 10      # Short-term VIX MA
VIX_LONG = 30       # Long-term VIX MA
ZSCORE_PERIOD = 180 # Z-Score calculation period

print(f"Backtest Period: {START_DATE} to {END_DATE}")
print(f"Initial Capital: ${INITIAL_CAPITAL:,}")

## Download Data

In [ ]:
print("Downloading SPY and VIX data...")

# Download SPY (S&P 500 ETF)
spy = yf.download('SPY', start=START_DATE, end=END_DATE, progress=True)

# Download VIX (CBOE Volatility Index)
vix = yf.download('^VIX', start=START_DATE, end=END_DATE, progress=True)

# Combine data
data = pd.DataFrame()
data['SPY_Close'] = spy['Close']
data['VIX_Close'] = vix['Close']

# Drop any rows with missing data
data = data.dropna()

print(f"\nDownloaded {len(data)} trading days of data")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")

# Display first few rows
data.head()

## Calculate VIX Z-Score Indicator

In [ ]:
print("Calculating VIX Z-Score indicator...")

# Calculate VIX moving averages
data['VIX_10'] = data['VIX_Close'].rolling(window=VIX_SHORT).mean()
data['VIX_30'] = data['VIX_Close'].rolling(window=VIX_LONG).mean()

# Calculate ratio (VIX30 / VIX10)
data['VIX_Ratio'] = data['VIX_30'] / data['VIX_10']

# Calculate Z-Score of the ratio
ratio_mean = data['VIX_Ratio'].rolling(window=ZSCORE_PERIOD).mean()
ratio_std = data['VIX_Ratio'].rolling(window=ZSCORE_PERIOD).std()

data['Z_Score'] = (data['VIX_Ratio'] - ratio_mean) / ratio_std

# Calculate WMA(2) of Z-Score (avgZv in ThinkScript)
weights = np.array([2, 1])
weights = weights / weights.sum()

data['avgZv'] = data['Z_Score'].rolling(window=2).apply(
    lambda x: np.sum(weights * x), raw=True
)

# Calculate moving averages of Z-Score for signal generation
data['Z_EMA3'] = data['Z_Score'].ewm(span=3, adjust=False).mean()
data['Z_SMA3'] = data['Z_Score'].rolling(window=3).mean()

# Drop rows with NaN values from calculations
data = data.dropna()

print(f"Z-Score calculation complete. {len(data)} valid data points.")

# Display statistics
print("\nZ-Score Statistics:")
print(data[['Z_Score', 'Z_SMA3']].describe())

## Generate Trading Signals

In [ ]:
print("Generating trading signals...")

signals = pd.DataFrame(index=data.index)
signals['Z_Score'] = data['Z_Score']
signals['Z_MA'] = data['Z_SMA3']  # Using SMA3 as the moving average
signals['Position'] = 0

# Track if we're in a position
in_position = False
positions = []

for i in range(1, len(signals)):
    current_z = signals['Z_Score'].iloc[i]
    prev_z = signals['Z_Score'].iloc[i-1]
    current_ma = signals['Z_MA'].iloc[i]
    prev_ma = signals['Z_MA'].iloc[i-1]
    
    if not in_position:
        # Entry condition: Z-score crosses above MA from below -1
        if (prev_z < -1 and prev_z < prev_ma and current_z > current_ma):
            positions.append(1)
            in_position = True
        else:
            positions.append(0)
    else:
        # Exit condition: Z-score crosses below MA from above 1
        if (prev_z > 1 and prev_z > prev_ma and current_z < current_ma):
            positions.append(0)
            in_position = False
        else:
            positions.append(1)  # Stay in position

# Add initial position (0) for first row
signals['Position'] = [0] + positions

# Generate trading orders (1 = buy, -1 = sell, 0 = hold)
signals['Signal'] = signals['Position'].diff()

num_buys = len(signals[signals['Signal'] == 1])
num_sells = len(signals[signals['Signal'] == -1])

print(f"Generated {num_buys} buy signals and {num_sells} sell signals")

# Display trade signals
trades = signals[signals['Signal'] != 0].copy()
trades['Action'] = trades['Signal'].apply(lambda x: 'BUY' if x == 1 else 'SELL')
trades['SPY_Price'] = data.loc[trades.index, 'SPY_Close']

print("\nFirst 10 trades:")
print(trades[['Action', 'SPY_Price', 'Z_Score', 'Z_MA']].head(10))

## Run Backtest

In [ ]:
print("Running backtest...")

portfolio = pd.DataFrame(index=signals.index)
portfolio['SPY_Close'] = data['SPY_Close']
portfolio['Position'] = signals['Position']
portfolio['Signal'] = signals['Signal']

# Calculate daily returns of SPY
portfolio['SPY_Returns'] = data['SPY_Close'].pct_change()

# Calculate strategy returns (only when in position)
portfolio['Strategy_Returns'] = (
    portfolio['Position'].shift(1) * portfolio['SPY_Returns']
)

# Calculate cumulative returns
portfolio['SPY_Cumulative'] = (1 + portfolio['SPY_Returns']).cumprod()
portfolio['Strategy_Cumulative'] = (1 + portfolio['Strategy_Returns']).cumprod()

# Calculate portfolio value
portfolio['Portfolio_Value'] = INITIAL_CAPITAL * portfolio['Strategy_Cumulative']
portfolio['SPY_Value'] = INITIAL_CAPITAL * portfolio['SPY_Cumulative']

print("Backtest complete!")

# Display recent portfolio values
portfolio[['SPY_Close', 'Position', 'Portfolio_Value', 'SPY_Value']].tail(10)

## Calculate Performance Metrics

In [ ]:
print("="*60)
print("BACKTEST RESULTS")
print("="*60)

# Basic metrics
final_value = portfolio['Portfolio_Value'].iloc[-1]
spy_final_value = portfolio['SPY_Value'].iloc[-1]

total_return = (final_value / INITIAL_CAPITAL - 1) * 100
spy_return = (spy_final_value / INITIAL_CAPITAL - 1) * 100

# Annualized metrics
years = len(portfolio) / 252  # Approximate trading days per year

annualized_return = (np.power(final_value / INITIAL_CAPITAL, 1/years) - 1) * 100
spy_annualized_return = (np.power(spy_final_value / INITIAL_CAPITAL, 1/years) - 1) * 100

# Volatility (annualized)
strategy_vol = portfolio['Strategy_Returns'].std() * np.sqrt(252) * 100
spy_vol = portfolio['SPY_Returns'].std() * np.sqrt(252) * 100

# Sharpe Ratio (assuming 0% risk-free rate)
sharpe_ratio = (annualized_return / strategy_vol) if strategy_vol > 0 else 0
spy_sharpe = (spy_annualized_return / spy_vol) if spy_vol > 0 else 0

# Maximum Drawdown
cumulative = portfolio['Strategy_Cumulative']
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min() * 100

spy_cumulative = portfolio['SPY_Cumulative']
spy_running_max = spy_cumulative.expanding().max()
spy_drawdown = (spy_cumulative - spy_running_max) / spy_running_max
spy_max_drawdown = spy_drawdown.min() * 100

# Win rate
winning_trades = len(portfolio[
    (portfolio['Signal'] == -1) &
    (portfolio['Strategy_Returns'] > 0)
])
total_trades = len(portfolio[portfolio['Signal'] == -1])
win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

# Time in market
time_in_market = (portfolio['Position'].sum() / len(portfolio)) * 100

print(f"\nPeriod: {data.index[0].date()} to {data.index[-1].date()}")
print(f"Trading Days: {len(portfolio)}")
print(f"Years: {years:.2f}")

print("\n" + "-"*60)
print("STRATEGY PERFORMANCE")
print("-"*60)
print(f"Initial Capital:        ${INITIAL_CAPITAL:,.2f}")
print(f"Final Value:            ${final_value:,.2f}")
print(f"Total Return:           {total_return:.2f}%")
print(f"Annualized Return:      {annualized_return:.2f}%")
print(f"Annualized Volatility:  {strategy_vol:.2f}%")
print(f"Sharpe Ratio:           {sharpe_ratio:.2f}")
print(f"Maximum Drawdown:       {max_drawdown:.2f}%")
print(f"Number of Trades:       {total_trades}")
print(f"Win Rate:               {win_rate:.2f}%")
print(f"Time in Market:         {time_in_market:.2f}%")

print("\n" + "-"*60)
print("BUY & HOLD SPY PERFORMANCE")
print("-"*60)
print(f"Final Value:            ${spy_final_value:,.2f}")
print(f"Total Return:           {spy_return:.2f}%")
print(f"Annualized Return:      {spy_annualized_return:.2f}%")
print(f"Annualized Volatility:  {spy_vol:.2f}%")
print(f"Sharpe Ratio:           {spy_sharpe:.2f}")
print(f"Maximum Drawdown:       {spy_max_drawdown:.2f}%")

print("\n" + "-"*60)
print("RELATIVE PERFORMANCE")
print("-"*60)
print(f"Excess Return:          {total_return - spy_return:.2f}%")
print(f"Excess Annual Return:   {annualized_return - spy_annualized_return:.2f}%")
print("="*60 + "\n")

## Visualize Results

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# Plot 1: Portfolio Value
axes[0].plot(portfolio.index, portfolio['Portfolio_Value'],
            label='Strategy', linewidth=2, color='blue')
axes[0].plot(portfolio.index, portfolio['SPY_Value'],
            label='Buy & Hold SPY', linewidth=2, color='gray', alpha=0.7)
axes[0].set_title('Portfolio Value Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Plot 2: VIX Z-Score
axes[1].plot(data.index, data['Z_Score'],
            label='Z-Score', linewidth=1.5, color='purple')
axes[1].plot(data.index, data['Z_SMA3'],
            label='SMA(3)', linewidth=1.5, color='orange')
axes[1].axhline(y=1, color='red', linestyle='--', alpha=0.5, label='±1')
axes[1].axhline(y=-1, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Mark buy/sell signals
buy_signals = signals[signals['Signal'] == 1]
sell_signals = signals[signals['Signal'] == -1]

axes[1].scatter(buy_signals.index, data.loc[buy_signals.index, 'Z_Score'],
               marker='^', color='green', s=100, label='Buy', zorder=5)
axes[1].scatter(sell_signals.index, data.loc[sell_signals.index, 'Z_Score'],
               marker='v', color='red', s=100, label='Sell', zorder=5)

axes[1].set_title('VIX Z-Score with Trading Signals', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Z-Score', fontsize=12)
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

# Plot 3: Position
axes[2].fill_between(portfolio.index, 0, portfolio['Position'],
                    alpha=0.3, color='green', label='In Position')
axes[2].set_title('Position Over Time', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Position (1=Long, 0=Cash)', fontsize=12)
axes[2].set_ylim(-0.1, 1.1)
axes[2].legend(loc='best')
axes[2].grid(True, alpha=0.3)

# Plot 4: Drawdown
cumulative = portfolio['Strategy_Cumulative']
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max * 100

axes[3].fill_between(portfolio.index, 0, drawdown,
                    alpha=0.3, color='red', label='Drawdown')
axes[3].set_title('Strategy Drawdown', fontsize=14, fontweight='bold')
axes[3].set_ylabel('Drawdown (%)', fontsize=12)
axes[3].set_xlabel('Date', fontsize=12)
axes[3].legend(loc='best')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Export Results to Google Drive (Optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Export trades to CSV
trades_export = trades[['Action', 'SPY_Price', 'Z_Score', 'Z_MA']].copy()
trades_export.to_csv('/content/drive/MyDrive/vix_zscore_trades.csv')
print("Trades exported to Google Drive: vix_zscore_trades.csv")

# Export full portfolio data
portfolio.to_csv('/content/drive/MyDrive/vix_zscore_portfolio.csv')
print("Portfolio exported to Google Drive: vix_zscore_portfolio.csv")

# Save the chart
fig.savefig('/content/drive/MyDrive/vix_zscore_backtest.png', dpi=300, bbox_inches='tight')
print("Chart saved to Google Drive: vix_zscore_backtest.png")

## Analyze Recent Signals

In [ ]:
# Show last 30 days of data
recent = data.tail(30).copy()
recent['Position'] = signals.loc[recent.index, 'Position']
recent['Signal'] = signals.loc[recent.index, 'Signal']

print("Last 30 days:")
print(recent[['SPY_Close', 'VIX_Close', 'Z_Score', 'Z_SMA3', 'Position']])

# Current position
current_position = portfolio['Position'].iloc[-1]
current_z = data['Z_Score'].iloc[-1]
current_ma = data['Z_SMA3'].iloc[-1]

print(f"\nCurrent Status:")
print(f"Position: {'LONG SPY' if current_position == 1 else 'CASH'}")
print(f"Z-Score: {current_z:.2f}")
print(f"Z-Score MA: {current_ma:.2f}")
print(f"SPY Price: ${data['SPY_Close'].iloc[-1]:.2f}")
print(f"VIX: {data['VIX_Close'].iloc[-1]:.2f}")